In [56]:
import numpy as np
import pandas as pd

from scipy.interpolate import PchipInterpolator
from scipy.optimize import minimize
from scipy import optimize
from scipy.stats import norm

In [57]:
df = pd.read_csv('Market_prices.csv')

In [58]:
maturities = np.sort(df["Maturity in years"].unique())
strikes = np.sort(df["strike"].unique())
DF_r = np.array([0.98566525, 0.96470589, 0.94480458, 0.92510071, 0.91545613, 0.88712149])
DF_d = np.array([0.98182105, 0.95639996, 0.92322802, 0.8891501 , 0.87359317, 0.82787945])
S0 = 278.54

In [59]:
log_dfr = np.log(DF_r)
log_dfd = np.log(DF_d)

interp_r = PchipInterpolator(maturities, log_dfr)
interp_d = PchipInterpolator(maturities, log_dfd)

def DF_r_curve(Ti):
    return np.exp(interp_r(Ti))

def DF_d_curve(Ti):
    return np.exp(interp_d(Ti))

In [60]:
def bs_price(F, K, T, sigma, DF_r):

    if sigma <= 0 or T <= 0:
        return DF_r * max(F - K, 0)

    d1 = (np.log(F/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    price = DF_r * (F * norm.cdf(d1) - K * norm.cdf(d2))

    return price

In [61]:
def bs_vega(F, K, T, sigma, DF_r):

    if sigma <= 0 or T <= 0:
        return 0

    d1 = (np.log(F/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))

    vega = DF_r * F * norm.pdf(d1) * np.sqrt(T)

    return vega

In [62]:
def initial_vol_guess(price, F, T):

    return np.sqrt(2*np.pi/T) * price / F

In [63]:
def implied_vol_newton(price, F, K, T, DF_r, tol=1e-8, max_iter=10):

    sigma = initial_vol_guess(price, F, T)

    for _ in range(max_iter):

        price_est = bs_price(F, K, T, sigma, DF_r)
        diff = price_est - price

        if abs(diff) < tol:
            return sigma

        vega = bs_vega(F, K, T, sigma, DF_r)

        if vega < 1e-8:
            break

        sigma = sigma - diff / vega

        if sigma <= 0:
            break

    return None

In [68]:
def implied_vol_brent(price, F, K, T, DF_r):

    f = lambda sigma: bs_price(F, K, T, sigma, DF_r) - price

    try:
        return brentq(f, 1e-6, 5.0)
    except:
        return np.nan

In [70]:
def implied_vol(price, F, K, T, DF_r):

    intrinsic = DF_r * max(F - K, 0)

    if price < intrinsic:
        return np.nan

    sigma = implied_vol_newton(price, F, K, T, DF_r)

    if sigma is not None:
        return sigma

    return implied_vol_brent(price, F, K, T, DF_r)

In [72]:
def compute_vol_surface(prices, strikes, maturities, F, DF_r):

    nK = len(strikes)
    nT = len(maturities)

    iv_surface = np.zeros((nT, nK))

    for i, T in enumerate(maturities):
        for j, K in enumerate(strikes):

            price = prices[i, j]

            if price <= 0:
                continue

            iv_surface[i, j] = implied_vol(price, F[i], K, T, DF_r[i])

    return iv_surface

In [74]:
F_array = []

for t in maturities:
    F = S0 * DF_d_curve(t) / DF_r_curve(t)
    F_array.append(F)

F_array

[277.4536641795985,
 276.1418247984368,
 272.17896497792174,
 267.715575371248,
 265.8026240665405,
 259.93907779530855]

In [76]:
price_matrix = np.zeros((len(maturities), len(strikes)))

for i, T in enumerate(maturities):
    
    df_T = df[np.isclose(df["Maturity in years"], T)].copy()
    
    # sort by strike
    df_T = df_T.sort_values("strike")
    
    # compute forward
    DF_r_T = DF_r_curve(T)
    DF_d_T = DF_d_curve(T)
    F = S0 * DF_d_T / DF_r_T
    
    for j, K in enumerate(strikes):
        
        row = df_T[df_T["strike"] == K]
        
        if len(row) == 0:
            continue
        
        # choose OTM price
        if K > F:
            price = row["mid_C"].values[0]
        else:
            price = row["mid_P"].values[0]
        
        if price <= 0:
            continue
        price_matrix[i, j] = price

iv_surface = compute_vol_surface(price_matrix, strikes, maturities, F_array, DF_r)

ivs = pd.DataFrame(iv_surface, columns = strikes, index = maturities)
ivs

,230,235,240,245,250,255,260,265,270,275,280,285,290,295,300,305,310,315,320
0.358333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.151211,0.227943,0.296987,0.286208,0.286225,0.267390,0.264869,0.265087,0.266873,0.264198,NaN
0.866667,NaN,NaN,NaN,NaN,NaN,NaN,0.084268,0.143994,0.193858,0.240238,0.295257,0.289358,0.285028,0.280984,0.277483,0.272715,0.270619,0.267016,0.264065
1.369444,NaN,0.0,NaN,0.000000,NaN,0.000000,0.146865,0.000000,0.223157,0.000000,0.335452,0.000000,0.325345,0.000000,0.315951,0.000000,0.308891,0.000000,0.301233
1.877778,NaN,NaN,NaN,0.058838,0.110326,0.147469,0.180989,0.212705,0.385970,0.379898,0.372150,0.369259,0.360763,0.355028,0.350912,0.346007,0.341043,0.337129,0.333901
2.130556,NaN,0.0,NaN,0.000000,0.127116,0.000000,0.191210,0.000000,0.398555,0.000000,0.385804,0.000000,0.373804,0.000000,0.364215,0.000000,0.355364,0.000000,0.346860
2.888889,NaN,0.0,0.102541,0.000000,0.162180,0.000000,0.450580,0.000000,0.433931,0.000000,0.422092,0.000000,0.409882,0.000000,0.397747,0.000000,0.389471,0.000000,0.379852


In [ ]:
import matplotlib.pyplot as plt

for i, T in enumerate(maturities):
    plt.plot(strikes, ivs.loc[T], label = f'T={T}')

plt.xlabel('strike')
plt.ylabel('IV')
plt.legend()


In [ ]:
# Step - 1
# Convert Strike → Log Moneyness
strikes = ivs.columns.values.astype(float)

for i, T in enumerate(ivs.index):
    F = F_array[i]

    row = ivs.loc[T]
    
    k = np.log( strikes / F )

In [ ]:
# Step - 2
# Convert IV → Total Variance

variance_surface = (ivs ** 2) * ivs.index.values[:, None]
variance_surface = variance_surface.replace(0, np.nan)
variance_surface

In [ ]:
# Step - 3
# Interpolate Missing Strikes
var_surface = variance_surface.copy()

for T in var_surface.index:
    row = var_surface.loc[T]
    row_k = pd.Series(row.values, index = k)
    row_k = row_k.sort_index()
    row_k = row_k.interpolate(method = 'linear')
    row_k = row_k.ffill().bfill()
    row_clean = pd.Series(row_k.values, index=row.index)

    var_surface.loc[T] = row_clean


In [ ]:
# Step - 4
# Convert back to Vol

iv_surface_clean = np.sqrt(var_surface.div(var_surface.index.values, axis = 0))
iv_surface_clean

In [ ]:
import numpy as np

X = iv_surface_clean.columns.astype(float).values   # strikes
Y = iv_surface_clean.index.values                   # maturities
Z = iv_surface_clean.values.astype(float)           # vols

X_grid, Y_grid = np.meshgrid(X, Y)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    X_grid,
    Y_grid,
    Z,
    cmap='viridis',
    edgecolor='none'
)

ax.set_xlabel("Strike")
ax.set_ylabel("Maturity")
ax.set_zlabel("Implied Volatility")

ax.set_title("ImlpliedVolatility Surface")

fig.colorbar(surf, shrink=0.5, aspect=10)

plt.show()